# Advanced: Multi-Strategy Dataset Merging with PAMOLA.CORE

**Goal**: Master all 3 join strategies using operation.execute()

**What you'll learn:**
- **Strategy 1**: Inner join (matching records only)
- **Strategy 2**: Left join (all left + matching right)
- **Strategy 3**: Outer join (all records from both)
- Calculate merge quality metrics
- Analyze data coverage and overlap
- Production deployment patterns

**Prerequisites:**
- Completed the simple notebook
- Understanding of SQL join concepts
- Familiarity with operation.execute() pattern
- Python 3.8+
- PAMOLA.CORE installed (auto-installs if needed)

---

## Step 1: Setup and Imports

**How to use:**
- Run this cell to import required libraries
- Auto-detects PAMOLA installation path (searches up 6 levels)
- If not found locally, auto-installs from GitHub

**What you'll see:**
- Detection message showing PAMOLA location or installation progress
- Import confirmation (✅ All imports successful!)
- Notebook start timestamp
- Project root and working directory paths

**Note:** First run may take 1-2 minutes if installing from GitHub

**Expected structure:**
```
PAMOLA/
├── pamola_core/          ← Auto-detected
└── examples/
    ├── data_examples/    ← Data directory
    └── transformations/
        └── 02_merge_datasets_advanced.ipynb  ← You are here
```

In [ ]:
import pandas as pd
import numpy as np
import json
import sys
import os
from pathlib import Path
from datetime import datetime
import time
from IPython.display import Image, display

print("🔍 Detecting PAMOLA installation...\n")

# Auto-detect project root (search up 6 levels for pamola_core/)
current_dir = Path.cwd()
project_root = current_dir
pamola_found = False

for level in range(6):
    if (project_root / 'pamola_core').exists():
        print(f"✓ Found pamola_core at level {level}: {project_root}")
        sys.path.insert(0, str(project_root))
        pamola_found = True
        break
    project_root = project_root.parent

# If not found locally, try to install from Git
if not pamola_found:
    print("⚠️  pamola_core not found in project structure")
    print("📦 Attempting to install from GitHub...\n")
    
    try:
        import subprocess
        
        # Install from Git repository
        install_cmd = [
            sys.executable, "-m", "pip", "install",
            "git+https://github.com/DGT-Network/PAMOLA.git@pre-epic3"
        ]
        
        result = subprocess.run(
            install_cmd,
            capture_output=True,
            text=True,
            check=True
        )
        
        print("✅ Successfully installed pamola_core from GitHub!")
        print("   Repository: https://github.com/DGT-Network/PAMOLA.git")
        print("   Branch: pre-epic3")
        
    except subprocess.CalledProcessError as e:
        print(f"❌ Installation failed!")
        print(f"   Error: {e.stderr}")
        raise RuntimeError(
            "Could not find pamola_core locally and installation from GitHub failed. "
            "Please install manually using:\n"
            "pip install git+https://github.com/DGT-Network/PAMOLA.git@pre-epic3"
        )
    except Exception as e:
        print(f"❌ Unexpected error during installation: {e}")
        raise

# Now try to import the required modules
try:
    from pamola_core.transformations.merging.merge_datasets_op import MergeDatasetsOperation
    from pamola_core.utils.ops.op_data_source import DataSource
    from pamola_core.utils.progress import HierarchicalProgressTracker
    from pamola_core.utils.tasks.task_reporting import TaskReporter
    from pamola_core.io.csv import read_csv
    
    print("✅ All imports successful!")
    print(f"📅 Notebook started: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print("=" * 80)
    print(f"📂 Project root:  {project_root.name}")
    print(f"📂 Working dir:   {current_dir.relative_to(project_root)}")
    print("=" * 80)
    
except ImportError as e:
    print(f"❌ Import failed: {e}")
    print("\n💡 Troubleshooting:")
    print("   1. Try restarting your Python kernel/notebook")
    print("   2. Verify installation: pip list | grep pamola")
    print("   3. Manual install: pip install git+https://github.com/DGT-Network/PAMOLA.git@pre-epic3")
    raise

## Step 2: Load Complex Dataset

**How to use:**
- Run to load or generate 1000-record datasets
- Auto-creates sample if files not found
- Review data structures before proceeding

**What you'll see:**
- Load status (from files or generated)
- Customers dataset (1000 records, 6 columns)
- Orders dataset (1500 records, 6 columns)
- Products dataset (800 records, 5 columns)
- Key field analysis showing overlap statistics

**Dataset features:**
- Realistic one-to-many relationships
- Partial overlaps (not all customers have orders)
- Cross-dataset references for complex joins

**Note:** Generated data auto-saves to examples/data_examples/ for reuse

In [ ]:
# Try to load from files first (in examples directory)
examples_dir = project_root / 'examples'
customers_path = examples_dir / 'data_examples' / 'merge_customers_complex.csv'
orders_path = examples_dir / 'data_examples' / 'merge_orders_complex.csv'
products_path = examples_dir / 'data_examples' / 'merge_products_complex.csv'

print(f"📂 Looking for data at: {examples_dir / 'data_examples'}\n")

# Generate or load customers
if customers_path.exists():
    print(f"📂 Loading customers from: {customers_path}")
    customers_df = read_csv(customers_path)
    print(f"✓ Loaded {len(customers_df)} customers from file")
else:
    print("📊 Generating customers dataset (1000 records)...")
    np.random.seed(42)
    n_customers = 1000
    
    customers_df = pd.DataFrame({
        'customer_id': range(1, n_customers + 1),
        'customer_name': [f'Customer_{i}' for i in range(1, n_customers + 1)],
        'email': [f'customer{i}@email.com' for i in range(1, n_customers + 1)],
        'city': np.random.choice(['New York', 'Los Angeles', 'Chicago', 'Houston', 'Phoenix', 
                                 'Philadelphia', 'San Antonio', 'San Diego', 'Dallas', 'San Jose'], n_customers),
        'segment': np.random.choice(['Enterprise', 'SMB', 'Startup', 'Individual'], n_customers),
        'join_date': pd.date_range('2020-01-01', periods=n_customers, freq='D')
    })
    
    try:
        customers_path.parent.mkdir(parents=True, exist_ok=True)
        customers_df.to_csv(customers_path, index=False)
        print(f"✓ Generated and saved to: {customers_path}")
    except PermissionError:
        print(f"⚠️  Cannot save to {customers_path} (file may be open)")

# Generate or load orders
if orders_path.exists():
    print(f"\n📂 Loading orders from: {orders_path}")
    orders_df = read_csv(orders_path)
    print(f"✓ Loaded {len(orders_df)} orders from file")
else:
    print("\n📊 Generating orders dataset (1500 records)...")
    n_orders = 1500
    
    # 80% of customers have orders, some have multiple
    customer_ids_with_orders = np.random.choice(range(1, 801), n_orders, replace=True)
    
    orders_df = pd.DataFrame({
        'order_id': range(1, n_orders + 1),
        'customer_id': customer_ids_with_orders,
        'product_id': np.random.randint(1, 801, n_orders),
        'order_date': pd.date_range('2023-01-01', periods=n_orders, freq='h'),
        'quantity': np.random.randint(1, 10, n_orders),
        'amount': np.random.randint(50, 5000, n_orders)
    })
    
    try:
        orders_df.to_csv(orders_path, index=False)
        print(f"✓ Generated and saved to: {orders_path}")
    except PermissionError:
        print(f"⚠️  Cannot save to {orders_path} (file may be open)")

# Generate or load products
if products_path.exists():
    print(f"\n📂 Loading products from: {products_path}")
    products_df = read_csv(products_path)
    print(f"✓ Loaded {len(products_df)} products from file")
else:
    print("\n📊 Generating products dataset (800 records)...")
    n_products = 800
    
    products_df = pd.DataFrame({
        'product_id': range(1, n_products + 1),
        'product_name': [f'Product_{i}' for i in range(1, n_products + 1)],
        'category': np.random.choice(['Electronics', 'Furniture', 'Clothing', 'Books', 'Toys'], n_products),
        'price': np.random.randint(10, 1000, n_products),
        'stock_quantity': np.random.randint(0, 500, n_products)
    })
    
    try:
        products_df.to_csv(products_path, index=False)
        print(f"✓ Generated and saved to: {products_path}")
    except PermissionError:
        print(f"⚠️  Cannot save to {products_path} (file may be open)")

print(f"\n📊 Dataset Overview:")
print("=" * 80)
print(f"  CUSTOMERS: {len(customers_df):,} records, {len(customers_df.columns)} columns")
print(f"  ORDERS:    {len(orders_df):,} records, {len(orders_df.columns)} columns")
print(f"  PRODUCTS:  {len(products_df):,} records, {len(products_df.columns)} columns")

print(f"\n🔍 Sample Records:")
print("\nCustomers (first 3):")
print(customers_df.head(3))
print("\nOrders (first 3):")
print(orders_df.head(3))
print("\nProducts (first 3):")
print(products_df.head(3))

print(f"\n🔑 Key Field Analysis:")
print(f"  Unique customer_ids in customers: {customers_df['customer_id'].nunique()}")
print(f"  Unique customer_ids in orders:    {orders_df['customer_id'].nunique()}")
print(f"  Customers with orders:            {len(set(customers_df['customer_id']) & set(orders_df['customer_id']))}")
print(f"  Customers without orders:         {len(set(customers_df['customer_id']) - set(orders_df['customer_id']))}")
print(f"  Orders per customer (avg):        {len(orders_df) / orders_df['customer_id'].nunique():.2f}")

print(f"\n💡 Perfect dataset for testing all 3 join strategies!")

## Step 3: Setup Shared Environment

**How to use:**
1. **CUSTOMIZE MERGE_CONFIG**: Edit merge parameters for each strategy
   - `left_dataset`: Dataset name (e.g., "customers")
   - `right_dataset`: Dataset to join (e.g., "orders")
   - `left_key` & `right_key`: Join fields
2. Run to validate fields and setup environment

**What you'll see:**
- Merge configuration validation (keys exist)
- Dataset statistics (record counts, unique keys)
- Task directory created (✓)
- TaskReporter initialized (✓)
- DataSource created with 3 datasets (✓)

**Merge configuration:**
- All strategies use customer_id as join key
- Left dataset: customers (1000 records)
- Right dataset: orders (1500 records)
- Relationship: one-to-many (one customer → many orders)

**Note:** All keys must exist in their respective datasets

In [ ]:
# Merge configuration - CUSTOMIZE THESE!
MERGE_CONFIG = {
    "left_dataset": "customers",
    "right_dataset": "orders",
    "left_key": "customer_id",
    "right_key": "customer_id",
    "relationship_type": "one-to-many"
}

# Validate keys exist
print("📋 Validating merge configuration...\n")
if MERGE_CONFIG["left_key"] not in customers_df.columns:
    raise ValueError(
        f"❌ Left key '{MERGE_CONFIG['left_key']}' not found in customers!\n"
        f"Available columns: {', '.join(customers_df.columns)}\n"
        f"Please update MERGE_CONFIG"
    )

if MERGE_CONFIG["right_key"] not in orders_df.columns:
    raise ValueError(
        f"❌ Right key '{MERGE_CONFIG['right_key']}' not found in orders!\n"
        f"Available columns: {', '.join(orders_df.columns)}\n"
        f"Please update MERGE_CONFIG"
    )

print("Merge Configuration:")
print(f"  ✓ Left dataset:  {MERGE_CONFIG['left_dataset']} ({len(customers_df)} records)")
print(f"    Join key:      {MERGE_CONFIG['left_key']} ({customers_df[MERGE_CONFIG['left_key']].nunique()} unique)")
print(f"\n  ✓ Right dataset: {MERGE_CONFIG['right_dataset']} ({len(orders_df)} records)")
print(f"    Join key:      {MERGE_CONFIG['right_key']} ({orders_df[MERGE_CONFIG['right_key']].nunique()} unique)")
print(f"\n  ✓ Relationship:  {MERGE_CONFIG['relationship_type']}")

# Configuration helper (production pattern)
def create_config_kwargs():
    return {
        "dataset_name": MERGE_CONFIG["left_dataset"]
    }

# Create task directory (in examples/data_examples)
examples_dir = project_root / 'examples'
task_dir = examples_dir / 'data_examples' / 'advanced_output'
os.makedirs(task_dir, exist_ok=True)
print(f"\n✓ Task directory: {task_dir}")

# Initialize TaskReporter
task_reporter = TaskReporter(
    task_id="advanced_001",
    task_type="multi_strategy",
    description="Multi-strategy dataset merging",
    report_path=task_dir
)
print("✓ TaskReporter initialized")

# Get config
kwargs = create_config_kwargs()
print(f"✓ Config kwargs ready")

# Create DataSource with all datasets
data_source = DataSource(
    dataframes={
        "customers": customers_df,
        "orders": orders_df,
        "products": products_df
    }
)
print("✓ DataSource created with 3 datasets")

print(f"\n✅ Shared environment ready for all strategies!")

## STRATEGY 1: Inner Join

**How to use:**
- Returns only matching records from both datasets
- Best for analysis requiring complete data
- Run to execute inner join strategy

**Key parameters:**
- `join_type='inner'`: Only matching records
- `relationship_type='one-to-many'`: One customer → many orders
- `suffixes=('_cust', '_ord')`: Handle duplicate column names

**What you'll see:**
- Configuration summary (datasets, keys, join type)
- Progress bar: validation → loading → merging → metrics → viz → save
- Completion message with execution time (1-3 seconds)
- Result statistics (matched records only)

**Expected results:**
- Output records ≈ 1500 (all orders with customer data)
- No NULL values in join keys
- Customers without orders excluded

**Validate:**
- ✅ Execution time <5 seconds
- ✅ Result count ≈ orders count
- ✅ All records have complete customer info
- ✅ Status = "completed"

**Note:** Inner join reduces data to intersection - use when you need complete records only

In [ ]:
print("\n" + "=" * 80)
print("🔄 STRATEGY 1: INNER JOIN")
print("=" * 80 + "\n")

# Setup progress tracker
tracker_s1 = HierarchicalProgressTracker(
    total=6,
    description="Strategy 1: Inner Join",
    unit="steps",
    track_memory=True,
    level=0
)

# Create operation
operation_s1 = MergeDatasetsOperation(
    # Core parameters
    left_dataset_name=MERGE_CONFIG["left_dataset"],
    right_dataset_name=MERGE_CONFIG["right_dataset"],
    left_key=MERGE_CONFIG["left_key"],
    right_key=MERGE_CONFIG["right_key"],
    
    # Strategy: Inner join
    join_type='inner',
    relationship_type=MERGE_CONFIG["relationship_type"],
    suffixes=('_cust', '_ord'),
    
    # Output configuration
    output_format='csv',
    
    # Processing settings
    use_cache=False,
    
    # Execution behavior
    generate_visualization=True,
    save_output=True,
    force_recalculation=False
)

print("✓ Strategy 1 configured")
print(f"  Join type:     {operation_s1.join_type}")
print(f"  Relationship:  {operation_s1.relationship_type}")
print(f"  Suffixes:      {operation_s1.suffixes}")
print(f"\n⚙️  Executing...\n")
start_time = time.time()

# Execute
result_s1 = operation_s1.execute(
    data_source=data_source,
    task_dir=task_dir / 'strategy1_inner',
    reporter=task_reporter,
    progress_tracker=tracker_s1,
    **kwargs
)

elapsed_s1 = time.time() - start_time
print(f"\n✅ Strategy 1 completed in {elapsed_s1:.2f}s")

# Load results (NEWEST file)
output_files_s1 = sorted(
    list((task_dir / 'strategy1_inner' / 'output').glob('*.csv')),
    key=lambda x: x.stat().st_mtime, reverse=True
)
if output_files_s1:
    result_df_s1 = pd.read_csv(output_files_s1[0])
    
    print(f"\n📊 Merge Results:")
    print(f"  Input (customers): {len(customers_df)} records")
    print(f"  Input (orders):    {len(orders_df)} records")
    print(f"  Output (inner):    {len(result_df_s1)} records")
    print(f"  Match rate:        {len(result_df_s1) / len(orders_df) * 100:.1f}% of orders")

## STRATEGY 2: Left Join

**How to use:**
- Returns all left records + matching right records
- Preserves all customers even without orders
- Run to execute left join strategy

**Key parameters:**
- `join_type='left'`: All left + matching right
- `relationship_type='one-to-many'`: One customer → many orders
- `suffixes=('_cust', '_ord')`: Handle duplicate column names

**What you'll see:**
- Configuration summary (datasets, keys, join type)
- Progress bar: validation → loading → merging → metrics → viz → save
- Completion message with execution time (1-3 seconds)
- Result statistics (all customers + their orders)

**Expected results:**
- Output records ≈ 1500+ (orders + customers without orders)
- NULL values in order fields for customers without orders
- All customers preserved

**Validate:**
- ✅ Execution time <5 seconds
- ✅ Result count ≥ orders count
- ✅ All customers included
- ✅ Status = "completed"

**Note:** Left join most common - keeps master dataset intact while enriching with details

In [ ]:
print("\n" + "=" * 80)
print("🔄 STRATEGY 2: LEFT JOIN")
print("=" * 80 + "\n")

tracker_s2 = HierarchicalProgressTracker(
    total=6,
    description="Strategy 2: Left Join",
    unit="steps",
    track_memory=True,
    level=0
)

operation_s2 = MergeDatasetsOperation(
    left_dataset_name=MERGE_CONFIG["left_dataset"],
    right_dataset_name=MERGE_CONFIG["right_dataset"],
    left_key=MERGE_CONFIG["left_key"],
    right_key=MERGE_CONFIG["right_key"],
    join_type='left',
    relationship_type=MERGE_CONFIG["relationship_type"],
    suffixes=('_cust', '_ord'),
    output_format='csv',
    use_cache=False,
    generate_visualization=True,
    save_output=True,
    force_recalculation=False
)

print("✓ Strategy 2 configured")
print(f"  Join type:     {operation_s2.join_type}")
print(f"  Relationship:  {operation_s2.relationship_type}")
print(f"  Suffixes:      {operation_s2.suffixes}")
print(f"\n⚙️  Executing...\n")
start_time = time.time()

result_s2 = operation_s2.execute(
    data_source=data_source,
    task_dir=task_dir / 'strategy2_left',
    reporter=task_reporter,
    progress_tracker=tracker_s2,
    **kwargs
)

elapsed_s2 = time.time() - start_time
print(f"\n✅ Strategy 2 completed in {elapsed_s2:.2f}s")

# Load results (NEWEST file)
output_files_s2 = sorted(
    list((task_dir / 'strategy2_left' / 'output').glob('*.csv')),
    key=lambda x: x.stat().st_mtime, reverse=True
)
if output_files_s2:
    result_df_s2 = pd.read_csv(output_files_s2[0])
    
    print(f"\n📊 Merge Results:")
    print(f"  Input (customers): {len(customers_df)} records")
    print(f"  Input (orders):    {len(orders_df)} records")
    print(f"  Output (left):     {len(result_df_s2)} records")
    
    # Count customers with/without orders
    if 'order_id' in result_df_s2.columns:
        with_orders = result_df_s2['order_id'].notna().sum()
        without_orders = result_df_s2['order_id'].isna().sum()
        print(f"  With orders:       {with_orders} records")
        print(f"  Without orders:    {without_orders} records")

## STRATEGY 3: Outer Join

**How to use:**
- Returns all records from both datasets
- Includes customers without orders AND orders without customers (orphans)
- Run to execute outer join strategy

**Key parameters:**
- `join_type='outer'`: All records from both
- `relationship_type='one-to-many'`: One customer → many orders
- `suffixes=('_cust', '_ord')`: Handle duplicate column names

**What you'll see:**
- Configuration summary (datasets, keys, join type)
- Progress bar: validation → loading → merging → metrics → viz → save
- Completion message with execution time (1-3 seconds)
- Result statistics (complete union of both datasets)

**Expected results:**
- Output records ≈ 1500+ (all orders + all customers)
- NULL values in customer fields for orphan orders
- NULL values in order fields for customers without orders

**Validate:**
- ✅ Execution time <5 seconds
- ✅ Result count = max coverage
- ✅ All data from both datasets preserved
- ✅ Status = "completed"

**Note:** Outer join provides complete view - useful for data quality analysis and finding orphans

In [ ]:
print("\n" + "=" * 80)
print("🔄 STRATEGY 3: OUTER JOIN")
print("=" * 80 + "\n")

tracker_s3 = HierarchicalProgressTracker(
    total=6,
    description="Strategy 3: Outer Join",
    unit="steps",
    track_memory=True,
    level=0
)

operation_s3 = MergeDatasetsOperation(
    left_dataset_name=MERGE_CONFIG["left_dataset"],
    right_dataset_name=MERGE_CONFIG["right_dataset"],
    left_key=MERGE_CONFIG["left_key"],
    right_key=MERGE_CONFIG["right_key"],
    join_type='outer',
    relationship_type=MERGE_CONFIG["relationship_type"],
    suffixes=('_cust', '_ord'),
    output_format='csv',
    use_cache=False,
    generate_visualization=True,
    save_output=True,
    force_recalculation=False
)

print("✓ Strategy 3 configured")
print(f"  Join type:     {operation_s3.join_type}")
print(f"  Relationship:  {operation_s3.relationship_type}")
print(f"  Suffixes:      {operation_s3.suffixes}")
print(f"\n⚙️  Executing...\n")
start_time = time.time()

result_s3 = operation_s3.execute(
    data_source=data_source,
    task_dir=task_dir / 'strategy3_outer',
    reporter=task_reporter,
    progress_tracker=tracker_s3,
    **kwargs
)

elapsed_s3 = time.time() - start_time
print(f"\n✅ Strategy 3 completed in {elapsed_s3:.2f}s")

# Load results (NEWEST file)
output_files_s3 = sorted(
    list((task_dir / 'strategy3_outer' / 'output').glob('*.csv')),
    key=lambda x: x.stat().st_mtime, reverse=True
)
if output_files_s3:
    result_df_s3 = pd.read_csv(output_files_s3[0])
    
    print(f"\n📊 Merge Results:")
    print(f"  Input (customers): {len(customers_df)} records")
    print(f"  Input (orders):    {len(orders_df)} records")
    print(f"  Output (outer):    {len(result_df_s3)} records")
    
    # Analyze outer join results
    if 'order_id' in result_df_s3.columns and 'customer_name' in result_df_s3.columns:
        matched = result_df_s3['order_id'].notna() & result_df_s3['customer_name'].notna()
        only_customers = result_df_s3['order_id'].isna() & result_df_s3['customer_name'].notna()
        only_orders = result_df_s3['order_id'].notna() & result_df_s3['customer_name'].isna()
        
        print(f"  Matched:           {matched.sum()} records")
        print(f"  Only customers:    {only_customers.sum()} records")
        print(f"  Only orders:       {only_orders.sum()} records (orphans)")

## Step 4: Strategy Comparison

**How to use:**
- Run AFTER all strategies complete
- Review performance and data coverage

**What you'll see:**
- Execution times (fastest to slowest)
- Total processing time
- Record counts comparison (inner < left ≤ outer)
- Data coverage analysis

**Strategy selection guide:**
- **Inner**: Fastest, only complete records, smallest dataset
- **Left**: Preserves master data, most common in practice
- **Outer**: Complete view, useful for QA and orphan detection

**Note:** Join type selection depends on business requirements, not just performance

In [ ]:
print("\n" + "=" * 80)
print("📊 STRATEGY COMPARISON")
print("=" * 80 + "\n")

print("⏱️  Execution Time:")
print(f"  Strategy 1 (Inner):  {elapsed_s1:6.2f}s")
print(f"  Strategy 2 (Left):   {elapsed_s2:6.2f}s")
print(f"  Strategy 3 (Outer):  {elapsed_s3:6.2f}s")
print(f"  Total:               {elapsed_s1 + elapsed_s2 + elapsed_s3:6.2f}s")

print(f"\n📊 Record Counts:")
print(f"  Input datasets:")
print(f"    Customers:  {len(customers_df):>6} records")
print(f"    Orders:     {len(orders_df):>6} records")
print(f"  Output results:")
if 'result_df_s1' in locals():
    print(f"    Inner join: {len(result_df_s1):>6} records ({len(result_df_s1)/len(orders_df)*100:.1f}% of orders)")
if 'result_df_s2' in locals():
    print(f"    Left join:  {len(result_df_s2):>6} records ({len(result_df_s2)/len(orders_df)*100:.1f}% of orders)")
if 'result_df_s3' in locals():
    print(f"    Outer join: {len(result_df_s3):>6} records ({len(result_df_s3)/len(orders_df)*100:.1f}% of orders)")

print(f"\n🎯 Data Coverage:")
print(f"  Inner: Most selective, only matching records")
print(f"  Left:  Preserves all customers")
print(f"  Outer: Complete view of all data")

## Step 5: Detailed Artifact Review

**How to use:**
- Run for comprehensive artifact inspection
- Focus on validation points per strategy

**What you'll see (per strategy):**
1. **Metrics JSON**: Match statistics, record counts, performance
2. **Output Data**: Sample of merged records
3. **Visualizations**: Join analysis charts (latest batch only)

**Key metrics to check:**
- Matched records count
- Records only in left/right
- Match percentage
- Field counts before/after

**Note:** Only newest files shown to avoid confusion from multiple runs

In [ ]:
# Review all strategies
strategy_dirs = [
    ('strategy1_inner', 'Strategy 1: Inner Join'),
    ('strategy2_left', 'Strategy 2: Left Join'),
    ('strategy3_outer', 'Strategy 3: Outer Join')
]

for dir_name, title in strategy_dirs:
    strategy_dir = task_dir / dir_name
    if not strategy_dir.exists():
        continue
        
    print("\n" + "=" * 80)
    print(f"📊 {title}")
    print("=" * 80)
    
    # 1. Metrics (NEWEST - with filtering)
    metrics_dir = strategy_dir / 'metrics'
    if metrics_dir.exists():
        metrics_files = list(metrics_dir.glob('*.json'))
        
        if metrics_files:
            # Exclude data_types files
            filtered = [f for f in metrics_files if not f.name.startswith("data_types")]

            if filtered:
                target_files = filtered
                print(f"✓ Found {len(filtered)} metrics file(s)")
            else:
                target_files = metrics_files
                print(f"⚠ Fallback to ALL {len(metrics_files)} file(s)")

            # Pick latest
            target_files.sort(key=lambda x: x.stat().st_mtime, reverse=True)
            latest_metrics_file = target_files[0]
            
            print(f"📄 {latest_metrics_file.name}\n")
            
            try:
                with open(latest_metrics_file, 'r') as f:
                    data = json.load(f)
                    metrics = data.get('metrics', {})
                
                # Display merge statistics
                print(f"   Merge Statistics:")
                if 'num_matched_records' in metrics:
                    print(f"     Matched records:     {metrics['num_matched_records']}")
                if 'num_only_in_left' in metrics:
                    print(f"     Only in left:        {metrics['num_only_in_left']}")
                if 'num_only_in_right' in metrics:
                    print(f"     Only in right:       {metrics['num_only_in_right']}")
                if 'match_percentage' in metrics:
                    print(f"     Match percentage:    {metrics['match_percentage']:.1f}%")
                
                # Display record counts
                print(f"\n   Record Counts:")
                if 'total_input_records' in metrics:
                    print(f"     Input records:       {metrics['total_input_records']}")
                if 'total_output_records' in metrics:
                    print(f"     Output records:      {metrics['total_output_records']}")
                
                # Performance
                if 'execution_time_seconds' in metrics:
                    print(f"\n   Duration: {metrics['execution_time_seconds']:.4f}s")
                    
            except Exception as e:
                print(f"   ⚠️  Error: {e}")
    
    # 2. Visualizations (NEWEST BATCH)
    viz_dir = strategy_dir / 'visualizations'
    if viz_dir.exists():
        viz_files = sorted(
            list(viz_dir.glob('*.png')),
            key=lambda x: x.stat().st_mtime, reverse=True
        )
        
        if viz_files:
            latest_time = viz_files[0].stat().st_mtime
            latest_batch = [
                f for f in viz_files 
                if abs(f.stat().st_mtime - latest_time) < 10
            ]
            
            print(f"\n📊 VISUALIZATIONS: {len(latest_batch)} files")
            for viz in latest_batch[:3]:  # Show first 3 only
                print(f"   📈 {viz.stem.replace('_', ' ').title()}")
                try:
                    display(Image(filename=str(viz)))
                except:
                    print(f"      (Display error)")

## Step 6: Calculate Data Quality Metrics

**How to use:**
- Run AFTER all strategies complete
- Calculates completeness and coverage

**What you'll see:**
- Original datasets: record counts and unique keys
- After merge: coverage percentages per strategy
- Completeness analysis (no NULLs vs with NULLs)
- Data quality score per join type

**Quality metrics:**
- Coverage: % of input records preserved
- Completeness: % of records with no NULL values
- Match rate: % of successful joins

**Note:** Inner join has highest completeness but lowest coverage

In [ ]:
print("\n" + "=" * 80)
print("📊 DATA QUALITY METRICS")
print("=" * 80 + "\n")

def calculate_coverage_metrics(result_df, left_df, right_df):
    """Calculate coverage and completeness metrics."""
    total_input = len(left_df) + len(right_df)
    coverage = len(result_df) / len(right_df) * 100  # Based on orders
    
    # Completeness: records with no NULL values
    complete_records = result_df.notna().all(axis=1).sum()
    completeness = complete_records / len(result_df) * 100
    
    return {
        'total_records': len(result_df),
        'coverage_pct': coverage,
        'complete_records': complete_records,
        'completeness_pct': completeness
    }

try:
    print(f"📊 ORIGINAL DATASETS:")
    print(f"   Customers: {len(customers_df)} records")
    print(f"   Orders:    {len(orders_df)} records")
    print(f"   Total:     {len(customers_df) + len(orders_df)} records")
    
    # Strategy 1: Inner Join
    if 'result_df_s1' in locals() and result_df_s1 is not None:
        metrics_s1 = calculate_coverage_metrics(result_df_s1, customers_df, orders_df)
        print(f"\n✨ STRATEGY 1 (Inner Join):")
        print(f"   Total records:     {metrics_s1['total_records']}")
        print(f"   Coverage:          {metrics_s1['coverage_pct']:.1f}% of orders")
        print(f"   Complete records:  {metrics_s1['complete_records']} ({metrics_s1['completeness_pct']:.1f}%)")
        print(f"   Quality:           High completeness, medium coverage")
    
    # Strategy 2: Left Join
    if 'result_df_s2' in locals() and result_df_s2 is not None:
        metrics_s2 = calculate_coverage_metrics(result_df_s2, customers_df, orders_df)
        print(f"\n✨ STRATEGY 2 (Left Join):")
        print(f"   Total records:     {metrics_s2['total_records']}")
        print(f"   Coverage:          {metrics_s2['coverage_pct']:.1f}% of orders")
        print(f"   Complete records:  {metrics_s2['complete_records']} ({metrics_s2['completeness_pct']:.1f}%)")
        print(f"   Quality:           Balanced completeness and coverage")
    
    # Strategy 3: Outer Join
    if 'result_df_s3' in locals() and result_df_s3 is not None:
        metrics_s3 = calculate_coverage_metrics(result_df_s3, customers_df, orders_df)
        print(f"\n✨ STRATEGY 3 (Outer Join):")
        print(f"   Total records:     {metrics_s3['total_records']}")
        print(f"   Coverage:          {metrics_s3['coverage_pct']:.1f}% of orders")
        print(f"   Complete records:  {metrics_s3['complete_records']} ({metrics_s3['completeness_pct']:.1f}%)")
        print(f"   Quality:           Maximum coverage, lower completeness")
    
    print(f"\n🎯 Recommendation: Choose join type based on use case")
    print(f"   - Inner: Analysis requiring complete records")
    print(f"   - Left:  Reporting on master dataset")
    print(f"   - Outer: Data quality audits")
    
except NameError:
    print("⚠️  Run all strategies first!")

## Step 7: Export Final Data

**How to use:**
- Run AFTER all strategies complete
- Exports merged datasets for each strategy

**What you'll see (per strategy):**
- Available columns list
- Export path
- Preview (first 5 rows)
- Statistics (record counts, completeness)

**Output structure:**
```
advanced_output/
├── strategy1_inner/
│   └── inner_join_data.csv
├── strategy2_left/
│   └── left_join_data.csv
└── strategy3_outer/
    └── outer_join_data.csv
```

**Note:** Files contain all merged columns - use for downstream analysis

In [ ]:
import os
from pathlib import Path

print("=" * 80)
print("📦 EXPORTING FINAL DATA FROM ALL STRATEGIES")
print("=" * 80)

# Base export directory
export_base_dir = task_dir
task_dir.mkdir(parents=True, exist_ok=True)

print(f"\n📂 Export directory: {task_dir}\n")

# ============================================================================
# STRATEGY 1: Inner Join
# ============================================================================
if 'result_df_s1' in locals() and result_df_s1 is not None:
    print("=" * 80)
    print("📊 STRATEGY 1: INNER JOIN")
    print("=" * 80)
    
    s1_dir = export_base_dir / 'strategy1_inner'
    s1_dir.mkdir(parents=True, exist_ok=True)
    
    # Save
    output_path_s1 = s1_dir / 'inner_join_data.csv'
    try:
        result_df_s1.to_csv(output_path_s1, index=False)
        print(f"\n✅ Saved: {output_path_s1}")
        print(f"   Rows: {len(result_df_s1):,}")
        print(f"   Columns: {len(result_df_s1.columns)}")
    except PermissionError:
        print(f"\n⚠️  Cannot save (file may be open)")
    
    # Preview
    print(f"\n📊 Preview:")
    print(result_df_s1.head())
    
    # Statistics
    print(f"\n📈 Statistics:")
    print(f"   Total records: {len(result_df_s1)}")
    print(f"   All matched: {result_df_s1.notna().all(axis=1).sum()} complete records")
else:
    print("\n⚠️  Strategy 1 data not available")

# ============================================================================
# STRATEGY 2: Left Join
# ============================================================================
if 'result_df_s2' in locals() and result_df_s2 is not None:
    print("\n" + "=" * 80)
    print("📊 STRATEGY 2: LEFT JOIN")
    print("=" * 80)
    
    s2_dir = export_base_dir / 'strategy2_left'
    s2_dir.mkdir(parents=True, exist_ok=True)
    
    output_path_s2 = s2_dir / 'left_join_data.csv'
    try:
        result_df_s2.to_csv(output_path_s2, index=False)
        print(f"\n✅ Saved: {output_path_s2}")
        print(f"   Rows: {len(result_df_s2):,}")
        print(f"   Columns: {len(result_df_s2.columns)}")
    except PermissionError:
        print(f"\n⚠️  Cannot save (file may be open)")
    
    print(f"\n📊 Preview:")
    print(result_df_s2.head())
    
    print(f"\n📈 Statistics:")
    print(f"   Total records: {len(result_df_s2)}")
    if 'order_id' in result_df_s2.columns:
        with_orders = result_df_s2['order_id'].notna().sum()
        without_orders = result_df_s2['order_id'].isna().sum()
        print(f"   With orders: {with_orders}")
        print(f"   Without orders: {without_orders}")
else:
    print("\n⚠️  Strategy 2 data not available")

# ============================================================================
# STRATEGY 3: Outer Join
# ============================================================================
if 'result_df_s3' in locals() and result_df_s3 is not None:
    print("\n" + "=" * 80)
    print("📊 STRATEGY 3: OUTER JOIN")
    print("=" * 80)
    
    s3_dir = export_base_dir / 'strategy3_outer'
    s3_dir.mkdir(parents=True, exist_ok=True)
    
    output_path_s3 = s3_dir / 'outer_join_data.csv'
    try:
        result_df_s3.to_csv(output_path_s3, index=False)
        print(f"\n✅ Saved: {output_path_s3}")
        print(f"   Rows: {len(result_df_s3):,}")
        print(f"   Columns: {len(result_df_s3.columns)}")
    except PermissionError:
        print(f"\n⚠️  Cannot save (file may be open)")
    
    print(f"\n📊 Preview:")
    print(result_df_s3.head())
    
    print(f"\n📈 Statistics:")
    print(f"   Total records: {len(result_df_s3)}")
    if 'order_id' in result_df_s3.columns and 'customer_name' in result_df_s3.columns:
        matched = (result_df_s3['order_id'].notna() & result_df_s3['customer_name'].notna()).sum()
        only_customers = (result_df_s3['order_id'].isna() & result_df_s3['customer_name'].notna()).sum()
        only_orders = (result_df_s3['order_id'].notna() & result_df_s3['customer_name'].isna()).sum()
        print(f"   Matched: {matched}")
        print(f"   Only customers: {only_customers}")
        print(f"   Only orders: {only_orders}")
else:
    print("\n⚠️  Strategy 3 data not available")

# ============================================================================
# SUMMARY
# ============================================================================
print("\n" + "=" * 80)
print("✅ EXPORT COMPLETE")
print("=" * 80)
print(f"\n📂 Files saved to: {export_base_dir}")

if 'elapsed_s1' in locals():
    total_time = elapsed_s1 + elapsed_s2 + elapsed_s3
    print(f"⏱️  Total time: {total_time:.2f}s")

## 🎯 Summary

**Accomplished:**
- ✅ 3 join strategies implemented and compared
- ✅ Data quality metrics calculated (coverage, completeness)
- ✅ Performance benchmarking completed
- ✅ Production-ready artifacts generated

**Key takeaways:**
- Inner join provides highest completeness (no NULLs) but excludes unmatched records
- Left join preserves master dataset while enriching with details - most common
- Outer join provides complete view for data quality analysis
- Join type selection impacts data coverage and completeness

**Strategy recommendations:**
- **Use Inner Join** when: Analysis requires complete records only, strict matching needed
- **Use Left Join** when: Master dataset must be preserved, standard enrichment scenario
- **Use Outer Join** when: Need complete view, finding orphans, data quality audits

**Best practices:**
- Always validate key uniqueness before merging
- Use suffixes to handle duplicate column names
- Check for orphan records in outer joins
- Monitor match percentages for data quality
- Document relationship type for maintainability

**Next steps:**
- Test with your own datasets
- Implement composite key joins (multiple fields)
- Add data validation rules
- Deploy to production pipeline

**Resources:**
- 📖 [PAMOLA Documentation](../../docs/)
- 🐙 [GitHub Repository](https://github.com/DGT-Network/PAMOLA)